# 01. Modele nieparametryczne

            Notebook odtwarza część nieparametryczną prezentacji: opis danych, cenzurowanie, estymator Kaplana-Meiera, tablicę aktuarialną, estymator Nelsona-Aalena, wygładzony hazard oraz testy dla podgrup.


## Import bibliotek i przygotowanie danych


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

# UWAGA:
# Plik z danymi powinien znajdować się najlepiej w tym samym folderze co notebook
# albo w podfolderze "data".
#
# Oczekiwana nazwa pliku:
# heart_failure_clinical_records_dataset.csv
#
# Kod nie używa bezwzględnych ścieżek Windows, dzięki czemu notebook powinien
# działać po przeniesieniu do innego katalogu roboczego.

DATA_FILENAME = "heart_failure_clinical_records_dataset.csv"


def find_data_file(filename: str = DATA_FILENAME) -> Path:
    """
    Szuka pliku danych w kilku typowych lokalizacjach względem katalogu roboczego
    i jego katalogów nadrzędnych.

    Priorytet:
    1. bieżący katalog roboczy,
    2. podfolder data w bieżącym katalogu,
    3. katalogi nadrzędne,
    4. podfolder data w katalogach nadrzędnych,
    5. rekurencyjne wyszukiwanie w bieżącym katalogu i katalogach nadrzędnych.
    """
    cwd = Path.cwd().resolve()

    search_roots = [cwd, *cwd.parents]

    direct_candidates = []
    for root in search_roots:
        direct_candidates.extend([
            root / filename,
            root / "data" / filename,
            root / "Data" / filename,
        ])

    for candidate in direct_candidates:
        if candidate.exists():
            return candidate.resolve()

    # Awaryjnie: szukaj rekurencyjnie w bieżącym katalogu i katalogach nadrzędnych.
    # To pozwala uruchomić notebook także z innego miejsca w strukturze projektu.
    for root in search_roots:
        matches = list(root.rglob(filename))
        if matches:
            return matches[0].resolve()

    raise FileNotFoundError(
        f"Nie znaleziono pliku danych: {filename}\n"
        "Umieść plik w tym samym folderze co notebook albo w podfolderze 'data'."
    )


DATA_PATH = find_data_file()

print(f"Wczytywany plik danych: {DATA_PATH}")

df = pd.read_csv(DATA_PATH).drop_duplicates().copy()

df["duration"] = df["time"].astype(float)
df["event"] = df["DEATH_EVENT"].astype(int)

df["age_60plus"] = (df["age"] >= 60).astype(int)
df["ef_low"] = (df["ejection_fraction"] < 35).astype(int)
df["creatinine_high"] = (df["serum_creatinine"] > 1.5).astype(int)
df["sodium_low"] = (df["serum_sodium"] < 135).astype(int)

df["log_cpk"] = np.log1p(df["creatinine_phosphokinase"])
df["log_platelets"] = np.log(df["platelets"])


def show(title, obj):
    print(f"\n{title}")
    print("-" * len(title))
    if isinstance(obj, pd.DataFrame):
        print(obj.to_string())
    elif isinstance(obj, pd.Series):
        print(obj.to_string())
    else:
        print(obj)


from lifelines import KaplanMeierFitter, NelsonAalenFitter
from lifelines.statistics import logrank_test

show(
    "Liczba obserwacji, zdarzeń i obserwacji cenzurowanych",
    pd.Series({
        "obserwacje": len(df),
        "zgony": int(df["event"].sum()),
        "cenzurowane": int((1 - df["event"]).sum()),
    })
)

## Statystyki opisowe i struktura cenzurowania


In [ ]:
quantitative = [
    "time", "age", "ejection_fraction", "serum_creatinine",
    "serum_sodium", "platelets", "creatinine_phosphokinase"
]
desc = df[quantitative].agg(["count", "mean", "std", "min", "median", "max"]).T.round(2)
show("Statystyki opisowe zmiennych ilościowych", desc)

binary_vars = ["DEATH_EVENT", "anaemia", "diabetes", "high_blood_pressure", "sex", "smoking", "ef_low", "creatinine_high", "sodium_low"]
binary_summary = pd.DataFrame({
    "N=0": [(df[v] == 0).sum() for v in binary_vars],
    "N=1": [(df[v] == 1).sum() for v in binary_vars],
    "udział 1": [df[v].mean() for v in binary_vars],
}, index=binary_vars).round(3)
show("Zmienne binarne i zbinaryzowane", binary_summary)

censoring = pd.DataFrame({
    "status": ["zgon", "cenzurowanie prawostronne"],
    "liczba": [int(df["event"].sum()), int((1 - df["event"]).sum())],
})
censoring["udział"] = (censoring["liczba"] / len(df)).round(3)
show("Struktura zdarzeń i cenzurowania", censoring)

fig, ax = plt.subplots(figsize=(6.8, 4.4))
ax.bar(censoring["status"], censoring["liczba"], color=["#C44E52", "#4C72B0"])
ax.set(title="Struktura zdarzeń i cenzurowania", ylabel="Liczba obserwacji")
for i, value in enumerate(censoring["liczba"]):
    ax.text(i, value + 3, str(value), ha="center")
plt.tight_layout()


## Rozkład czasu obserwacji


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.hist(df["duration"], bins=18, color="#4C72B0", edgecolor="white")
ax.set(title="Rozkład czasu obserwacji", xlabel="Czas obserwacji (dni)", ylabel="Liczba pacjentów")
plt.tight_layout()


## Estymator Kaplana-Meiera


In [ ]:
kmf = KaplanMeierFitter(label="Kaplan-Meier")
kmf.fit(df["duration"], event_observed=df["event"])

km_table = kmf.event_table.reset_index().rename(columns={
    "event_at": "czas", "at_risk": "n_ryzyka", "observed": "zgony", "censored": "cenzurowane"
})
km_surv = kmf.survival_function_.reset_index().rename(columns={"timeline": "czas", "Kaplan-Meier": "S(t)"})
km_fragment = km_table.merge(km_surv, on="czas", how="left").head(12).round(4)
show("Fragment tabeli Kaplana-Meiera", km_fragment)

fig, ax = plt.subplots(figsize=(7.2, 4.8))
kmf.plot_survival_function(ci_show=True, ax=ax, color="#4C72B0")
ax.set(title="Funkcja przeżycia Kaplana-Meiera", xlabel="Czas obserwacji (dni)", ylabel="S(t)")
plt.tight_layout()


## Tablica aktuarialna


In [ ]:
def life_table(data, breaks=(0, 30, 60, 90, 120, 150, 180, 210, 240, 285)):
    rows, survival = [], 1.0
    for start, end in zip(breaks[:-1], breaks[1:]):
        at_risk = int((data["duration"] >= start).sum())
        events = int(((data["duration"] >= start) & (data["duration"] < end) & (data["event"] == 1)).sum())
        censored = int(((data["duration"] >= start) & (data["duration"] < end) & (data["event"] == 0)).sum())
        effective = at_risk - censored / 2
        q = events / effective if effective > 0 else np.nan
        survival *= 1 - q if pd.notna(q) else 1
        hazard = events / (effective * (end - start)) if effective > 0 else np.nan
        rows.append([f"[{start},{end})", at_risk, events, censored, effective, q, survival, hazard])
    return pd.DataFrame(rows, columns=["przedział", "n_początkowe", "zgony", "cenzurowane", "n_efektywne", "q", "S_act", "h"])

actuarial = life_table(df)
show("Tablica aktuarialna", actuarial.round(4))

mids = np.arange(len(actuarial))
fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.step(mids, actuarial["S_act"], where="mid", color="#4C72B0", lw=2)
ax.scatter(mids, actuarial["S_act"], color="#4C72B0")
ax.set_xticks(mids)
ax.set_xticklabels(actuarial["przedział"], rotation=35, ha="right")
ax.set(title="Tablica aktuarialna: funkcja przeżycia", xlabel="Przedział czasu", ylabel="S(t)")
plt.tight_layout()


## Nelson-Aalen i wygładzona funkcja hazardu


In [ ]:
naf = NelsonAalenFitter(label="Nelson-Aalen")
naf.fit(df["duration"], event_observed=df["event"])

fig, ax = plt.subplots(figsize=(7.2, 4.8))
naf.plot_cumulative_hazard(ci_show=True, ax=ax, color="#4C72B0")
ax.set(title="Skumulowany hazard Nelsona-Aalena", xlabel="Czas obserwacji (dni)", ylabel="H(t)")
plt.tight_layout()

bandwidth = 50
naf2 = NelsonAalenFitter().fit(df["duration"], df["event"])
hazard = naf2.smoothed_hazard_(bandwidth)
hazard_ci = naf2.smoothed_hazard_confidence_intervals_(bandwidth)
show("Wygładzony hazard Nelsona-Aalena: pierwsze wartości", hazard.head(10).round(6))

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(hazard.index, hazard.iloc[:, 0], color="steelblue", lw=2, label="h(t)")
ax.fill_between(hazard_ci.index, hazard_ci.iloc[:, 0], hazard_ci.iloc[:, 1],
                alpha=0.2, color="steelblue", label="95% przedział ufności")
ax.set(xlabel="Czas (liczba dni)", ylabel="h(t)", title="Szacunkowa funkcja hazardu")
ax.legend()
plt.tight_layout()


## Krzywe Kaplana-Meiera dla podgrup


In [ ]:
subgroup_vars = ["sex", "anaemia", "diabetes", "high_blood_pressure", "smoking", "ef_low", "creatinine_high", "sodium_low"]
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5), sharey=True)
for ax, var in zip(axes.flat, subgroup_vars):
    for value, color in [(0, "#4C72B0"), (1, "#C44E52")]:
        mask = df[var] == value
        KaplanMeierFitter(label=f"{var}={value}").fit(df.loc[mask, "duration"], df.loc[mask, "event"]).plot_survival_function(
            ax=ax, ci_show=False, color=color, lw=1.9
        )
    ax.set_title(var)
    ax.set_xlabel("dni")
    ax.set_ylabel("S(t)")
fig.suptitle("Kaplan-Meier dla zmiennych binarnych i zbinaryzowanych", y=1.02)
plt.tight_layout()


## Testy log-rank i Wilcoxona


In [ ]:
rows = []
for var in ["sex", "anaemia", "diabetes", "high_blood_pressure", "smoking", "ef_low", "creatinine_high"]:
    a = df[df[var] == 0]
    b = df[df[var] == 1]
    lr = logrank_test(a["duration"], b["duration"], event_observed_A=a["event"], event_observed_B=b["event"])
    wx = logrank_test(a["duration"], b["duration"], event_observed_A=a["event"], event_observed_B=b["event"], weightings="wilcoxon")
    rows.append([
        var, len(a), int(a["event"].sum()), len(b), int(b["event"].sum()),
        lr.test_statistic, lr.p_value, wx.test_statistic, wx.p_value
    ])
tests = pd.DataFrame(rows, columns=["zmienna", "N0", "zgony0", "N1", "zgony1", "logrank_chi2", "logrank_p", "wilcoxon_chi2", "wilcoxon_p"])
show("Wyniki testów nieparametrycznych", tests.round(4))


## Wnioski kontrolne

            Notebook odtwarza elementy nieparametryczne prezentacji na podstawie danych wejściowych: tabele opisowe, cenzurowanie, krzywe przeżycia, hazardy i testy różnic między podgrupami. Progi zmiennych zbinaryzowanych są jawnie zdefiniowane w sekcji przygotowania danych.
